## 02 — Feature Collections

In the last lesson we read a GeoJSON file and pulled data out of it. Every feature we looked at was a `Point`.

This lesson focuses on the **container** — the `FeatureCollection` — and answers two questions:

1. What geometry types can live inside one?
2. How do you add a new feature to an existing collection?

## Valid Geometry Types

A `FeatureCollection` can hold any mix of these geometry types:

| Type | Shape | Coordinates structure |
|---|---|---|
| `Point` | a single location | `[lon, lat]` |
| `LineString` | a path between two or more points | `[[lon, lat], [lon, lat], ...]` |
| `MultiLineString` | multiple disconnected paths | `[[[lon, lat], ...], [[lon, lat], ...]]` |
| `Polygon` | a closed area | `[[[lon, lat], ..., [lon, lat]]]` — first and last point must match |
| `MultiPolygon` | multiple disconnected areas | one more nesting level out |

This course works with **Points, LineStrings, MultiLineStrings, and Polygons**.

All of these can live in the same `FeatureCollection` at the same time. The `type` field on the geometry tells you which one you have.

## Building a Feature Collection from Scratch

The minimum valid structure is just a dict with two keys:

In [1]:
collection = {
    "type": "FeatureCollection",
    "features": []
}

print(collection)

{'type': 'FeatureCollection', 'features': []}


## Adding a Point Feature

A `Point` marks a single location. Coordinates are `[longitude, latitude]`.

In [2]:
point_feature = {
    "type": "Feature",
    "properties": {
        "name": "Meteor Crater",
        "description": "Impact site in Arizona"
    },
    "geometry": {
        "type": "Point",
        "coordinates": [-111.0225, 35.0272]   # [lon, lat]
    }
}

collection["features"].append(point_feature)
print(f"Features in collection: {len(collection['features'])}")

Features in collection: 1


## Adding a LineString Feature

A `LineString` is an ordered list of points that form a path. It takes at least two coordinate pairs.

In [3]:
line_feature = {
    "type": "Feature",
    "properties": {
        "name": "Approach Path",
        "description": "Estimated trajectory of the meteor"
    },
    "geometry": {
        "type": "LineString",
        "coordinates": [
            [-111.5, 35.5],    # start
            [-111.2, 35.3],
            [-111.0225, 35.0272]  # end — the crater
        ]
    }
}

collection["features"].append(line_feature)
print(f"Features in collection: {len(collection['features'])}")

Features in collection: 2


## Adding a Polygon Feature

A `Polygon` is a closed shape. The rules:

- Coordinates are wrapped in **an extra list** — `[[ ... ]]` — because a polygon can have interior rings (holes)
- The first and last coordinate pair **must be identical** to close the ring

In [4]:
polygon_feature = {
    "type": "Feature",
    "properties": {
        "name": "Blast Zone",
        "description": "Approximate affected area around the crater"
    },
    "geometry": {
        "type": "Polygon",
        "coordinates": [[          # outer ring — note the double bracket
            [-111.1, 35.1],
            [-110.9, 35.1],
            [-110.9, 34.9],
            [-111.1, 34.9],
            [-111.1, 35.1]         # closes the ring — same as the first point
        ]]
    }
}

collection["features"].append(polygon_feature)
print(f"Features in collection: {len(collection['features'])}")

Features in collection: 3


## Inspecting the Mixed Collection

All three geometry types now live in the same collection. You can loop over them and check `geometry["type"]` to handle each one differently.

In [5]:
for feature in collection["features"]:
    name = feature["properties"]["name"]
    geom_type = feature["geometry"]["type"]
    print(f"{geom_type:<16} {name}")

Point            Meteor Crater
LineString       Approach Path
Polygon          Blast Zone


## Adding Features from an Existing File

You can also pull features out of a loaded file and append them into a collection — useful for merging datasets.

In [6]:
import json
from pathlib import Path

meteorites = json.loads(Path("data/meteorites.geojson").read_text())

# append the first three meteorite features into our collection
for feature in meteorites["features"][:3]:
    collection["features"].append(feature)

print(f"Features in collection: {len(collection['features'])}")

# confirm the mix of geometry types
for feature in collection["features"]:
    print(feature["geometry"]["type"], "-", feature["properties"]["name"])

Features in collection: 6
Point - Meteor Crater
LineString - Approach Path
Polygon - Blast Zone
Point - Aarhus
Point - Abee
Point - Adzhi-Bogdo (stone)


## Saving a Collection Back to a File

Once you've built or modified a collection, write it back out with `json.dumps`.

In [7]:
output_path = Path("data/my_collection.geojson")
output_path.write_text(json.dumps(collection, indent=2))

print(f"Saved to {output_path}")

Saved to data\my_collection.geojson


## Exercise A

Add a `MultiLineString` feature to `collection` representing two separate survey transects. Give it a name and description in `properties`, and fill in two 2-point paths in `coordinates`.

Refer to the geometry types table above for the correct coordinate nesting structure.

In [8]:
multiline_feature = {
    "type": "Feature",
    "properties": {
        "name": "Survey Lines",
        "description": "Two separate survey transects"
    },
    "geometry": {
        "type": "MultiLineString",
        "coordinates": [
            [[-111.3, 35.2], [-111.4, 35.1]], # path 1
            [[-110.8, 34.8], [-110.7, 34.7]]  # path 2
        ]
    }
}

collection["features"].append(multiline_feature)
print(f"Features in collection: {len(collection['features'])}")

Features in collection: 7


## Exercise B

Write a function `count_coordinates(feature) -> int` that returns the total number of coordinate pairs in any feature — it must handle `Point`, `LineString`, and `Polygon` geometry types.

Test it by printing the coordinate count for each feature in the collection.

In [9]:
def count_coordinates(feature) -> int:
    geom = feature["geometry"]
    g_type = geom["type"]
    coords = geom["coordinates"]
    
    if g_type == "Point":
        return 1
    elif g_type == "LineString":
        return len(coords)
    elif g_type == "Polygon" or g_type == "MultiLineString":
        # Polygons are lists of rings; MultiLineStrings are lists of lines
        # We sum the length of each internal list
        return sum(len(inner_list) for inner_list in coords)
    return 0

for f in collection["features"]:
    name = f["properties"]["name"]
    geom_type = f["geometry"]["type"]
    print(f"{geom_type:<16} {name:<30} coords: {count_coordinates(f)}")

Point            Meteor Crater                  coords: 1
LineString       Approach Path                  coords: 3
Polygon          Blast Zone                     coords: 5
Point            Aarhus                         coords: 1
Point            Abee                           coords: 1
Point            Adzhi-Bogdo (stone)            coords: 1
MultiLineString  Survey Lines                   coords: 4


## Exercise C

Load `meteorites.geojson`, filter to only meteorites with `mass > 50000`, and save the result as a new `FeatureCollection` to `data/heavy_meteorites.geojson`. Print how many passed the filter.

In [10]:
import json
from pathlib import Path

# Load data
meteorites = json.loads(Path("data/meteorites.geojson").read_text())

# Filter features
heavy_features = []
for f in meteorites["features"]:
    # Safely convert mass to float for comparison
    mass_val = f["properties"].get("mass")
    try:
        if mass_val is not None and float(mass_val) > 50000:
            heavy_features.append(f)
    except ValueError:
        continue

# Build new collection
heavy_collection = {
    "type": "FeatureCollection",
    "features": heavy_features
}

# Save
output_path = Path("data/heavy_meteorites.geojson")
output_path.write_text(json.dumps(heavy_collection, indent=2))

print(f"Filtered {len(heavy_features)} meteorites over 50,000g.")

Filtered 434 meteorites over 50,000g.


---

## Check Your Understanding

You have this partial feature:

```python
feature = {
    "type": "Feature",
    "properties": {"name": "Exclusion Zone"},
    "geometry": {
        "type": "Polygon",
        "coordinates": ???
    }
}
```

Fill in the `coordinates` value for a small square polygon with these four corners (in order):
`[-100.0, 40.0]`, `[-99.0, 40.0]`, `[-99.0, 39.0]`, `[-100.0, 39.0]`

Two things to get right: the nesting level, and closing the ring.

```python
# Answer
feature["geometry"]["coordinates"] = [[
    [-100.0, 40.0], 
    [-99.0, 40.0], 
    [-99.0, 39.0], 
    [-100.0, 39.0], 
    [-100.0, 40.0]  
]]
```

## Next

In [02 — Viewing GeoJSON](../02-Viewing_GeoJSON/00-Geojson.io.ipynb), we take the collections we've been building and put them on a map.